In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

def estimate_period_fft(series, default_period=24):
    """
    FFT를 활용하여 시계열 데이터에서 가장 지배적인 주기를 동적으로 추정합니다.
    - 스펙트럼 왜곡 방지를 위해 1차적으로 선형 디트렌딩(Linear Detrending)을 수행합니다.
    """
    n = len(series)
    t = np.arange(n)
    
    # 1. FFT 전처리: 선형 추세 및 평균(DC 성분) 제거
    p = np.polyfit(t, series, 1)
    detrended = series - np.polyval(p, t)
    detrended = detrended - np.mean(detrended)
    
    # 2. FFT 수행 및 진폭 계산
    fft_vals = np.fft.rfft(detrended)
    frequencies = np.fft.rfftfreq(n)
    magnitudes = np.abs(fft_vals)
    
    # 3. 0Hz(상수항)를 제외하고 진폭이 가장 큰 인덱스 탐색
    if len(magnitudes) > 1:
        dominant_idx = np.argmax(magnitudes[1:]) + 1 
        dominant_freq = frequencies[dominant_idx]
        
        if dominant_freq > 0:
            period = int(np.round(1.0 / dominant_freq))
            # 현실적인 시계열 분석 바운더리 설정 (최소 주기 2, 최대 주기 전체 길이의 절반)
            if 2 <= period <= n // 2:
                return period
                
    return default_period

def calculate_quadrant_metrics(series):
    """
    FFT 기반으로 동적 주기를 추출한 뒤, STL 분해를 수행하여 F_T, F_S 지표를 산출합니다.
    """
    # 동적 주기 추정
    estimated_p = estimate_period_fft(series)
    
    # STL 분해 수행 (추정 주기가 유효한지 재확인)
    if len(series) < 2 * estimated_p:
        estimated_p = max(2, len(series) // 2 - 1)
        
    try:
        res = STL(series, period=estimated_p, robust=True).fit()
        T, S, R = res.trend, res.seasonal, res.resid
        
        var_R = np.var(R)
        var_TR = np.var(T + R)
        var_SR = np.var(S + R)
        
        F_T = max(0, 1 - (var_R / var_TR if var_TR > 0 else 1))
        F_S = max(0, 1 - (var_R / var_SR if var_SR > 0 else 1))
    except Exception:
        # 분해 실패 시 예외 처리 (정상성 노이즈로 간주)
        F_T, F_S = 0.0, 0.0
        
    return F_T, F_S, estimated_p

def generate_extensive_synthetic_dataset(n_samples_per_q=50, length=500):
    """
    4사분면 검증을 위해 통계적 특성이 다양한 복잡한 가상 시계열 데이터를 대량 생성합니다.
    """
    np.random.seed(42)
    data_records = []
    t = np.arange(length)
    
    for i in range(n_samples_per_q):
        # 무작위 고유 주기 부여 (12시간, 24시간, 48시간 단위 등 무작위화)
        rand_p1 = np.random.choice([12, 24, 36, 48])
        rand_p2 = rand_p1 * 2  # 다중 주기성 구현용
        
        # ----------------------------------------------------
        # Q1: Trending & Seasonal (강한 추세 + 강한 계절성)
        # ----------------------------------------------------
        # 선형+이차 추세와 복합 사인파 결합, 미미한 노이즈
        slope = np.random.uniform(0.1, 0.4) * np.random.choice([1, -1])
        curve = np.random.uniform(1e-5, 5e-5) * (t**2) * np.random.choice([1, -1])
        amp1 = np.random.uniform(8, 25)
        amp2 = np.random.uniform(2, 8)
        noise_std = np.random.uniform(0.5, 1.5)
        
        q1_series = (slope * t + curve) + \
                    (amp1 * np.sin(2 * np.pi * t / rand_p1)) + \
                    (amp2 * np.cos(2 * np.pi * t / rand_p2)) + \
                    np.random.normal(0, noise_std, length)
                    
        ft, fs, p_est = calculate_quadrant_metrics(q1_series)
        data_records.append({'True_Quadrant': 'Q1', 'Pattern': 'Trend+MultiSeason', 'F_T': ft, 'F_S': fs, 'True_P': rand_p1, 'Est_P': p_est})
        
        # ----------------------------------------------------
        # Q2: Pure Seasonal (약한 추세 + 강한 계절성)
        # ----------------------------------------------------
        # 평균 0 바운더리 내 감쇄 진동 주기성 또는 복합 주기성, 미미한 노이즈
        amp1 = np.random.uniform(10, 30)
        amp2 = np.random.uniform(4, 12)
        noise_std = np.random.uniform(0.5, 2.0)
        
        # 주기가 명확한 복합 신호
        q2_series = (amp1 * np.sin(2 * np.pi * t / rand_p1)) + \
                    (amp2 * np.sin(2 * np.pi * t / rand_p2 + np.pi/4)) + \
                    np.random.normal(0, noise_std, length)
                    
        ft, fs, p_est = calculate_quadrant_metrics(q2_series)
        data_records.append({'True_Quadrant': 'Q2', 'Pattern': 'Pure_MultiSeason', 'F_T': ft, 'F_S': fs, 'True_P': rand_p1, 'Est_P': p_est})
        
        # ----------------------------------------------------
        # Q3: Stationary / Noise (약한 추세 + 약한 계절성)
        # ----------------------------------------------------
        # 화이트 노이즈, 강한 AR(1)/MA(1) 프로세스 (계절성과 무관한 노이즈성 상관관계)
        noise_type = np.random.choice(['WN', 'AR'])
        if noise_type == 'WN':
            q3_series = np.random.normal(0, np.random.uniform(5, 15), length)
        else:
            # 약한 자기회귀 시계열 생성
            phi = np.random.uniform(0.4, 0.7)
            q3_series = np.zeros(length)
            innovations = np.random.normal(0, 5, length)
            q3_series[0] = innovations[0]
            for idx in range(1, length):
                q3_series[idx] = phi * q3_series[idx-1] + innovations[idx]
                
        ft, fs, p_est = calculate_quadrant_metrics(q3_series)
        data_records.append({'True_Quadrant': 'Q3', 'Pattern': f'Stationary_{noise_type}', 'F_T': ft, 'F_S': fs, 'True_P': 'None', 'Est_P': p_est})
        
        # ----------------------------------------------------
        # Q4: Pure Trending (강한 추세 + 약한 계절성)
        # ----------------------------------------------------
        # 구조적 단절(Structural Break)이 있는 급격한 추세, 랜덤 워크(Drift 포함), 지수 성장
        trend_type = np.random.choice(['Break', 'RandomWalk', 'Exponential'])
        noise_std = np.random.uniform(0.5, 1.5)
        
        if trend_type == 'Break':
            # 특정 시점에 평균이 급변하는 구조적 단절 패턴
            break_point = length // 2
            q4_series = np.random.normal(0, noise_std, length)
            q4_series[break_point:] += np.random.uniform(50, 150)
        elif trend_type == 'RandomWalk':
            # 누적 합 기반의 랜덤 워크 (Drift 포함)
            drift = np.random.uniform(0.1, 0.5) * np.random.choice([1, -1])
            q4_series = np.cumsum(np.random.normal(drift, 2, length))
        else:
            # 지수 형태로 우상향하는 추세
            q4_series = np.exp(np.linspace(0, np.random.uniform(3, 5), length)) + np.random.normal(0, noise_std, length)
            
        ft, fs, p_est = calculate_quadrant_metrics(q4_series)
        data_records.append({'True_Quadrant': 'Q4', 'Pattern': f'PureTrend_{trend_type}', 'F_T': ft, 'F_S': fs, 'True_P': 'None', 'Est_P': p_est})
        
    return pd.DataFrame(data_records)

In [ ]:
# ==========================================
# 1. 시뮬레이션 데이터 생성 (각 사분면별 50개, 총 200개 시퀀스)
# ==========================================
print("💡 다양한 패턴의 합성 시계열 데이터를 대량 생성 및 오라클 라우팅 분석 중...")
df_exp_results = generate_extensive_synthetic_dataset(n_samples_per_q=100, length=500)


💡 다양한 패턴의 합성 시계열 데이터를 대량 생성 및 오라클 라우팅 분석 중...


In [ ]:
# ==========================================
# 2. 결과 시각화 (사분면 맵 플롯팅)
# ==========================================
plt.figure(figsize=(11, 9))
quadrant_colors = {'Q1': '#d63031', 'Q2': '#0984e3', 'Q3': '#2ed573', 'Q4': '#e67e22'}
quadrant_names = {
    'Q1': 'Q1: Composite Regime',
    'Q2': 'Q2: Seasonal Regime',
    'Q3': 'Q3: Stationary Regime',
    'Q4': 'Q4: Trending Regime'
}

for q_label, group in df_exp_results.groupby('True_Quadrant'):
    plt.scatter(group['F_T'], group['F_S'], 
                label=quadrant_names[q_label], 
                color=quadrant_colors[q_label], 
                alpha=0.75, edgecolors='k', s=70)

# 임계치 가이드라인 (tau = 0.64)
plt.axvline(x=0.64, color='#2d3436', linestyle='--', linewidth=2)
plt.axhline(y=0.64, color='#2d3436', linestyle='--', linewidth=2)

# 각 사분면 영역 텍스트 표기
plt.text(0.82, 0.82, 'Q1\n(Composite)', fontsize=13, fontweight='bold', ha='center', color='#b21f1f')
plt.text(0.18, 0.82, 'Q2\n(Seasonal)', fontsize=13, fontweight='bold', ha='center', color='#0652dd')
plt.text(0.18, 0.18, 'Q3\n(Stationary)', fontsize=13, fontweight='bold', ha='center', color='#1b8a43')
plt.text(0.82, 0.18, 'Q4\n(Trending)', fontsize=13, fontweight='bold', ha='center', color='#d35400')

plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.xlabel('Trend Strength ($F_T$)', fontsize=13, fontweight='bold')
plt.ylabel('Seasonal Strength ($F_S$)', fontsize=13, fontweight='bold')
plt.title('Empirical Validation of TSFM Gating Oracle using FFT-based STL', fontsize=15, fontweight='bold', pad=15)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, alpha=0.25)
plt.show()

# ==========================================
# 3. 오라클 분류 정확도 결산 리포트 출력
# ==========================================
print("\n" + "="*60)
print("     [오라클 검증 통계 리포트 (임계치: 0.64)]")
print("="*60)

total_success = 0
for q_label in ['Q1', 'Q2', 'Q3', 'Q4']:
    sub_df = df_exp_results[df_exp_results['True_Quadrant'] == q_label]
    
    # 0.64 문턱값 기준 분류 성공 여부 판단 수식
    if q_label == 'Q1':
        success_condition = (sub_df['F_T'] >= 0.64) & (sub_df['F_S'] >= 0.64)
    elif q_label == 'Q2':
        success_condition = (sub_df['F_T'] < 0.64) & (sub_df['F_S'] >= 0.64)
    elif q_label == 'Q3':
        success_condition = (sub_df['F_T'] < 0.64) & (sub_df['F_S'] < 0.64)
    elif q_label == 'Q4':
        success_condition = (sub_df['F_T'] >= 0.64) & (sub_df['F_S'] < 0.64)
        
    success_count = len(sub_df[success_condition])
    total_success += success_count
    accuracy = (success_count / len(sub_df)) * 100
    
    print(f"▶ {quadrant_names[q_label]}")
    print(f"  - 생성 패턴 예시: {sub_df['Pattern'].unique()}")
    print(f"  - 평균 지표값  : Mean F_T = {sub_df['F_T'].mean():.3f} | Mean F_S = {sub_df['F_S'].mean():.3f}")
    
    # 주기 매칭 검증 (Q1, Q2에 대해서만 추출 성과 체크)
    if q_label in ['Q1', 'Q2']:
        same_p_count = len(sub_df[sub_df['True_P'] == sub_df['Est_P']])
        p_match_rate = (same_p_count / len(sub_df)) * 100
        print(f"  - FFT 주기 추정 정확도: {p_match_rate:.1f}%")
        
    print(f"  - 사분면 매핑 성공률 : [ {success_count} / {len(sub_df)} ] ({accuracy:.1f}%)")
    print("-" * 60)

print(f"■ 전체 통합 오라클 라우팅 정확도: {(total_success / len(df_exp_results))*100:.2f}%")
print("="*60)

In [8]:
df_exp_results[df_exp_results['Pattern'] == 'Pure_MultiSeason']

,True_Quadrant,Pattern,F_T,F_S,True_P,Est_P
1,Q2,Pure_MultiSeason,0.541203,0.982762,36,36
5,Q2,Pure_MultiSeason,0.387251,0.978907,48,50
9,Q2,Pure_MultiSeason,0.639546,0.950752,48,50
13,Q2,Pure_MultiSeason,0.709380,0.939634,36,36
17,Q2,Pure_MultiSeason,0.639633,0.964430,36,36
...,...,...,...,...,...,...
381,Q2,Pure_MultiSeason,0.441230,0.978623,48,50
385,Q2,Pure_MultiSeason,0.738139,0.968738,36,36
389,Q2,Pure_MultiSeason,0.711254,0.987819,12,12
393,Q2,Pure_MultiSeason,0.654740,0.951844,48,50
